# 02. 결측률 프로파일 및 오프라인 변수 제외

## tl;dr

`merged_data_ko.csv`는 113,935행 × 39열이다. 오프라인 변수 5개는 각각 111,873건(98.19%)이 결측이고 나머지 34개 변수의 결측률은 0%다. 기준 모델링 데이터에서는 **오프라인 변수이면서 결측률 95% 이상인 5개 컬럼만 제외**한다.

## Context & Methods

### Key Assumptions

- `merge_ko`의 한 행은 한 배치의 특정 발효시간 관측값이다.
- 컬럼명에 `오프라인`이 포함된 변수만 오프라인 측정값으로 분류한다.
- 결측률 95% 이상인 오프라인 변수는 일괄 대체하면 대부분의 값을 인위적으로 생성하게 되므로 기준 모델에서 제외한다.
- 원본 `merge_ko`는 보존하며 결과 컬럼은 이번 단계에서 삭제하지 않는다. 모델링 단계에서 feature와 target을 별도로 분리해야 한다.

## Data

한글 컬럼명과 정렬이 적용된 interim 데이터를 `merge_ko`로 불러온다.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'interim' / 'merged_data_ko.csv').exists():
            return candidate
    raise FileNotFoundError('data/interim/merged_data_ko.csv를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / 'data' / 'interim' / 'merged_data_ko.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / '이상치 결측치'
PROFILE_PATH = OUTPUT_DIR / '결측률_프로파일.csv'
CLEAN_PATH = OUTPUT_DIR / 'merged_data_no_offline.csv'

merge_ko = pd.read_csv(DATA_PATH)
print(f'입력: {DATA_PATH}')
print(f'크기: {merge_ko.shape[0]:,}행 × {merge_ko.shape[1]}열')

## Results

### 1. 전체 컬럼 결측률 프로파일

In [ ]:
OFFLINE_MISSING_THRESHOLD = 95.0

missing_profile = pd.DataFrame({
    '원래순서': range(1, merge_ko.shape[1] + 1),
    '컬럼명': merge_ko.columns,
    '자료형': merge_ko.dtypes.astype(str).to_numpy(),
    '전체행수': len(merge_ko),
    '유효값수': merge_ko.notna().sum().to_numpy(),
    '결측수': merge_ko.isna().sum().to_numpy(),
    '결측률(%)': merge_ko.isna().mean().mul(100).round(2).to_numpy(),
    '고유값수': merge_ko.nunique(dropna=True).to_numpy(),
})

missing_profile['변수구분'] = '일반'
offline_mask = missing_profile['컬럼명'].str.contains('오프라인', na=False)
missing_profile.loc[offline_mask, '변수구분'] = '오프라인'

exclude_mask = offline_mask & (missing_profile['결측률(%)'] >= OFFLINE_MISSING_THRESHOLD)
missing_profile['제외결정'] = '유지'
missing_profile.loc[exclude_mask, '제외결정'] = '제외'
missing_profile['결정근거'] = '오프라인 변수가 아님'
missing_profile.loc[offline_mask & ~exclude_mask, '결정근거'] = '오프라인 변수이나 결측률 기준 미만'
missing_profile.loc[exclude_mask, '결정근거'] = (
    f'오프라인 변수이며 결측률 {OFFLINE_MISSING_THRESHOLD:.0f}% 이상'
)

missing_profile = missing_profile.sort_values(
    ['결측률(%)', '원래순서'], ascending=[False, True]
).reset_index(drop=True)

PROFILE_PATH.parent.mkdir(parents=True, exist_ok=True)
missing_profile.to_csv(PROFILE_PATH, index=False, encoding='utf-8-sig')
display(missing_profile)

### 2. 오프라인 변수 제외 결정 및 적용

결정표에서 `제외`로 표시된 컬럼만 제거한다. 원본 `merge_ko`는 변경하지 않고 `merge_ko_no_offline`을 새로 만든다.

In [ ]:
offline_decision = missing_profile.loc[
    missing_profile['변수구분'].eq('오프라인'),
    ['컬럼명', '전체행수', '유효값수', '결측수', '결측률(%)', '제외결정', '결정근거'],
].copy()
excluded_offline_columns = offline_decision.loc[
    offline_decision['제외결정'].eq('제외'), '컬럼명'
].tolist()

merge_ko_no_offline = merge_ko.drop(columns=excluded_offline_columns).copy()

# 현재 데이터 기준 검증
assert len(excluded_offline_columns) == 5
assert merge_ko_no_offline.shape == (len(merge_ko), merge_ko.shape[1] - 5)
assert not any('오프라인' in column for column in merge_ko_no_offline.columns)
assert merge_ko_no_offline.isna().sum().sum() == 0

CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
merge_ko_no_offline.to_csv(CLEAN_PATH, index=False, encoding='utf-8-sig')

display(offline_decision)
print(f'제외 컬럼 수: {len(excluded_offline_columns)}개')
print(f'제외 컬럼: {excluded_offline_columns}')
print(f'결과 크기: {merge_ko_no_offline.shape[0]:,}행 × {merge_ko_no_offline.shape[1]}열')
print(f'프로파일 저장: {PROFILE_PATH}')
print(f'오프라인 제외 데이터 저장: {CLEAN_PATH}')

### 3. 저장 결과 재검증

In [ ]:
saved_profile = pd.read_csv(PROFILE_PATH)
saved_clean = pd.read_csv(CLEAN_PATH)

assert saved_profile.shape == (39, 11)
assert saved_clean.shape == merge_ko_no_offline.shape
assert saved_clean.columns.tolist() == merge_ko_no_offline.columns.tolist()
assert saved_clean.isna().sum().sum() == 0

print('저장 결과 검증 통과')

## Takeaways

- 오프라인 5개 변수의 결측률은 모두 98.19%이므로 기준 모델에서는 제외한다.
- 나머지 34개 변수에는 결측값이 없다.
- 오프라인 데이터 자체가 잘못된 것은 아니다. 측정 시점이 제한된 희소 데이터이므로, 이후 오프라인 분석이나 별도 저빈도 모델을 만들 때는 원본 `merge_ko`에서 다시 사용할 수 있다.
- 결과 컬럼과 결함 컬럼은 아직 남아 있으므로 모델링 전에 target과 feature를 반드시 분리한다.